In [1]:
# =============== 纯计算版：无 GUI / 无绘图依赖 ===============
from copy import deepcopy

# 项目配置（路径等）
from config import DATA_DIR, INPUT_DIR,OUTPUT_DIR  # 如有 OUTPUT_DIR 也可一起导入

# 业务模块（仅计算相关）
import genaric2.tegnode as tegnode
import draw.read_snap_xml as read_snap_xml
import draw.basic_functio.write2xml as write2xml
import draw.basic_functio.transnodes as transnodes
import draw.basic_functio.conflict_link as conflict_link

# 基础参数
N = 36
P = 18

backend (before pyplot): QtAgg
backend (after pyplot): qtagg


In [2]:
file_in = DATA_DIR
# xml_file = r"DATA_DIR\station_visible_satellites_648_1d_real.xml"
xml_file = DATA_DIR / "station_visible_satellites_648_1d_real.xml"


In [3]:
from typing import Dict, Any

def slice_group_data(raw_group_data, start, end):
    """
    从 raw_group_data 中裁剪时间区间 [basicSa, end)
    """
    return {
        step: raw_group_data[step]
        for step in range(start, end)
        if step in raw_group_data
    }


In [4]:

# 只做一次：解析大区间
RAW_START, RAW_END = 8485, 30152
raw_group_data = read_snap_xml.parse_xml_group_data(xml_file, RAW_START, RAW_END)

#下面是图变换的。
#

In [5]:
from draw.pyqt_draw.pyqt_main2 import SatelliteViewer


import genaric2.tegnode as tegnode

In [20]:
start1 =13057

end1 = 14680

file_path = INPUT_DIR / f"interplane_links_{start1}_{end1}.xml"
nodes1 = write2xml.xml_to_nodes(file_path, tegnode.tegnode)
# nodes1  = write2xml.xml_to_nodes(  rf"E:\研究生\研究进展\工作记录\实验记录\interplane_links_{start_ts}_{end_ts}.xml", tegnode.tegnode)


In [21]:
start2 =14680

end2 = 16296

file_path = INPUT_DIR / f"interplane_links_{start2}_{end2}.xml"
nodes2 = write2xml.xml_to_nodes(file_path, tegnode.tegnode)
# nodes2  = write2xml.xml_to_nodes(  rf"E:\研究生\研究进展\工作记录\实验记录\interplane_links_{start_ts}_{end_ts}.xml", tegnode.tegnode)
#
# nodes2  = write2xml.xml_to_nodes( = INPUT_DIR+'interplane_links_{start_ts}_{end_ts}.xml"', tegnode.tegnode)

In [22]:

start3 =16296

end3 = 18396

file_path = INPUT_DIR / f"interplane_links_{start3}_{end3}.xml"
nodes3 = write2xml.xml_to_nodes(file_path, tegnode.tegnode)
#
# nodes3  = write2xml.xml_to_nodes(  rf"E:\研究生\研究进展\工作记录\实验记录\interplane_links_{start_ts}_{end_ts}.xml", tegnode.tegnode)

In [23]:
totalnode_raw = {**nodes1, **nodes2,**nodes3 }


In [24]:
totalnode = deepcopy(totalnode_raw)

In [25]:

# 用法（左闭右开或双闭任选）
start_ts = start1
# end_ts   = 86399Q
end_ts   = end3

group_data = slice_group_data(raw_group_data, start_ts, end_ts)


In [40]:
# # ====================== 读取数据 ======================
# # xml_file = r"E:\Data\station_visible_satellites_648_1d_real.xml"
#
#
# xml_file = DATA_DIR / "station_visible_satellites_648_1d_real.xml"
#
# # 解析 XML 得到 group_data，结构：{time_step: {'groups': {...}}}
# group_data = read_snap_xml.parse_xml_group_data(xml_file, start1, end3)


In [ ]:
rev_group_data,offset = read_snap_xml.modify_group_data(group_data, N=36, groupid=4)

## 处理冲突边我们现在就需要将汇总的nodes，重新进行一次冲突边处理,我们一开始就是分而治之，所以，这里我们需要将汇总的nodes，重新进行一次冲突边处理

在进行操作的时候，一定要留意，我们代码中，有可能会修改totalnode内容的。下面的作用就是将总的totalnode转为边，这样我们就获得整个区间的情况

In [26]:
import draw.basic_functio.transnodes as transnodes
totalnode_comple = transnodes.transnodes(totalnode)

In [27]:
time_2_build=60
import draw.basic_functio.conflict_link as conflict_link
# 这个是目前最新的，仍然有小bug，但是暂时不修了，等后面再修
raw_edges_by_step,pending_edges,modifynodes = conflict_link.get_no_conflict_link_nodes3(totalnode_comple , start1, end3,time_2_build,N,P)


In [28]:

# 这里，我们需要将nodes 信息转化为edge信息,注意，这个edge也只有inter-edge，并且是原始
import draw.basic_functio.inter_edge2nodes as inter_edge2nodes

# all_inter_edge = inter_edge2nodes.trans_nodes2edges(totalnode,P,N)

all_inter_edge = raw_edges_by_step

In [29]:
# 转化为inter-edge信息后，我们可以通过绘图来查看是否是争取的

# ====================== 绘图初始化 ======================
# 1) QApplication 实例（全局唯一）
app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])

# 2) 确保 viewer 有全局引用，避免 GC 回收导致崩溃
if not hasattr(sys.modules[__name__], "_viewer_list"):
    _viewer_list = []



In [30]:

# 3) 创建并配置 viewer
viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("onestep")
viewer.resize(1200, 700)
# viewer.edges_by_step = all_inter_edge
# viewer.pending_links_by_step =pending_edges

viewer.edges_by_step = raw_edges_by_step
viewer.pending_links_by_step =pending_edges
viewer.show()



肉眼检查后没问题，记录下一段区间。

In [31]:


start, end = start2, end2
filternodes = {}

for step in range(start, end):
    for x in range(P):
        for y in range(N):
            key = (x, y, step)
            if key in modifynodes:        # 只保留存在的键
                filternodes[key] = modifynodes[key]


再次检查

In [32]:

# 这里，我们需要将nodes 信息转化为edge信息,注意，这个edge也只有inter-edge，并且是原始
import draw.basic_functio.inter_edge2nodes as inter_edge2nodes
all_inter_edge = inter_edge2nodes.trans_nodes2edges(filternodes,P,N)
time_2_build = 60
# 这里，我们还需要将pending edges 信息也加入到all_inter_edge 中
pending_edges = inter_edge2nodes.trans_nodes2_pendingedges(filternodes, start1, end3, time_2_build,P, N)


In [33]:
# 转化为inter-edge信息后，我们可以通过绘图来查看是否是争取的

# ====================== 绘图初始化 ======================
# 1) QApplication 实例（全局唯一）
app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])

# 2) 确保 viewer 有全局引用，避免 GC 回收导致崩溃
if not hasattr(sys.modules[__name__], "_viewer_list"):
    _viewer_list = []


# 3) 创建并配置 viewer
viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("flitter node grpah two-auth")
viewer.resize(1200, 700)
viewer.edges_by_step = all_inter_edge
viewer.pending_links_by_step =pending_edges
viewer.show()



In [22]:
_viewer_list.append(viewer)


检查无误后，存入到文件夹内

In [ ]:

file_path = INPUT_DIR / f"onestep/interplane_links_{start}_{end}.xml"


write2xml.nodes_to_xml2(
    filternodes,
   file_path
)


下面是最后一段的生成代码

In [45]:
start, end = start3, end3

filternodes = {}

for step in range(start, end):
    for x in range(P):
        for y in range(N):
            key = (x, y, step)
            if key in modifynodes:        # 只保留存在的键
                filternodes[key] = modifynodes[key]


In [46]:

file_path = INPUT_DIR / f"onestep/interplane_links_{start}_{end}.xml"


write2xml.nodes_to_xml2(
    filternodes,
   file_path
)